In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize': 'large',
        'xtick.labelsize': 'medium',
        'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import pytz

NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFRCalSpreadRV -- SOFR Futures Calendar Spread RV Screener

This notebook demonstrates the full RV screener across the Q12 SOFR contract ladder:
- **Calendar spreads** (3M, 6M, 9M, 12M gaps)
- **Butterflies / microflies** (3M, 6M, 9M, 12M gaps)  
- **Double butterflies** (3M, 6M gaps)

For each structure: level, 1d change, roll/carry, vol, z-score, risk-adjusted roll.

In [ ]:
from BT.signals.sfr_cal_spread_rv import (
    SFRCalSpreadRVConfig,
    StructureType,
    STRUCTURE_LABELS,
    build_snapshot,
    load_rate_panel,
    compute_structure,
    compute_zscore_ts,
    analyze_specific_fly,
)

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

## 1. Load Rate Panel

Fetch EOD rates for Q12 SOFR contracts via `BARCHART_STIRF-RL`.

In [ ]:
config = SFRCalSpreadRVConfig(
    n_contracts=12,
    zscore_window=40,
    vol_window=20,
)

curve_mdp = IRSwapsMDP(source=config.source)
ts_builder = TimeseriesBuilder()

start = NYC.localize(datetime.datetime(2025, 10, 1, 18, 0))
end = 'live'

rates = load_rate_panel(
    config, start=start, end=end,
    curve_mdp=curve_mdp, ts_builder=ts_builder,
)
print(f'Rate panel: {rates.shape[0]} dates x {rates.shape[1]} contracts')
print(f'Ladder: {list(rates.columns)}')
print(f'Date range: {rates.index[0]} to {rates.index[-1]}')
rates.tail()

## 2. Build Full Screener Snapshot

In [ ]:
snap = build_snapshot(config, rates_panel=rates)
print(f'Snapshot as of: {snap.as_of}')
print(f'Structures computed: {[st.value for st in snap.structures.keys()]}')

## 3. Strip (Outright Rates)

In [ ]:
if StructureType.STRIP in snap.structures:
    strip_data = snap.structures[StructureType.STRIP]
    strip_df = strip_data.to_dataframe()
    display(strip_df)

## 4. Calendar Spreads

Convention: back - front, so positive = upward sloping.

In [ ]:
for st in [StructureType.SPD_3M, StructureType.SPD_6M, StructureType.SPD_12M]:
    if st not in snap.structures:
        continue
    data = snap.structures[st]
    if not data.labels:
        continue
    print(f'\n=== {STRUCTURE_LABELS[st]} ===')
    s = data.summary()
    if 'peak' in s and 'trough' in s:
        print(f'  Peak: {s["peak"]["label"]} = {s["peak"]["value"]:+.1f} bp')
        print(f'  Trough: {s["trough"]["label"]} = {s["trough"]["value"]:+.1f} bp')
    if 'max_chg' in s and 'min_chg' in s:
        print(f'  Max Chg: {s["max_chg"]["label"]} = {s["max_chg"]["value"]:+.1f} bp')
        print(f'  Min Chg: {s["min_chg"]["label"]} = {s["min_chg"]["value"]:+.1f} bp')
    display(data.to_dataframe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for idx, st in enumerate([StructureType.SPD_3M, StructureType.SPD_6M, StructureType.SPD_9M, StructureType.SPD_12M]):
    ax = axes[idx // 2][idx % 2]
    if st not in snap.structures or not snap.structures[st].labels:
        ax.set_title(f'{STRUCTURE_LABELS[st]} -- no data')
        continue
    data = snap.structures[st]
    x = range(len(data.labels))
    ax.plot(x, data.levels, 'o-', color='tab:cyan', label='Latest', linewidth=2, markersize=5)
    ax.bar(x, data.changes, color=['tab:green' if c >= 0 else 'tab:red' for c in data.changes], alpha=0.4, label='1d Chg')
    ax.set_xticks(x)
    ax.set_xticklabels(data.labels, rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f'{STRUCTURE_LABELS[st]} Curve (bp)', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Butterflies (Microflies)

Standard [1, -2, 1] weights. Positive = wings expensive vs belly.

In [ ]:
for st in [StructureType.FLY_3M, StructureType.FLY_6M, StructureType.FLY_12M]:
    if st not in snap.structures:
        continue
    data = snap.structures[st]
    if not data.labels:
        continue
    print(f'\n=== {STRUCTURE_LABELS[st]} ===')
    s = data.summary()
    if 'peak' in s and 'trough' in s:
        print(f'  Peak: {s["peak"]["label"]} = {s["peak"]["value"]:+.1f} bp')
        print(f'  Trough: {s["trough"]["label"]} = {s["trough"]["value"]:+.1f} bp')
    display(data.to_dataframe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for idx, st in enumerate([StructureType.FLY_3M, StructureType.FLY_6M, StructureType.FLY_9M, StructureType.FLY_12M]):
    ax = axes[idx // 2][idx % 2]
    if st not in snap.structures or not snap.structures[st].labels:
        ax.set_title(f'{STRUCTURE_LABELS[st]} -- no data')
        continue
    data = snap.structures[st]
    x = range(len(data.labels))
    ax.plot(x, data.levels, 'o-', color='tab:cyan', label='Latest', linewidth=2, markersize=5)
    ax.bar(x, data.changes, color=['tab:green' if c >= 0 else 'tab:red' for c in data.changes], alpha=0.4, label='1d Chg')
    ax.set_xticks(x)
    ax.set_xticklabels(data.labels, rotation=45, ha='right', fontsize=7)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f'{STRUCTURE_LABELS[st]} Curve (bp)', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Double Butterflies

Weights: [1, -3, 3, -1]. Spikes indicate kinks in the curve.

In [ ]:
for st in [StructureType.DFLY_3M, StructureType.DFLY_6M]:
    if st not in snap.structures:
        continue
    data = snap.structures[st]
    if not data.labels:
        continue
    print(f'\n=== {STRUCTURE_LABELS[st]} ===')
    s = data.summary()
    if 'peak' in s and 'trough' in s:
        print(f'  Peak: {s["peak"]["label"]} = {s["peak"]["value"]:+.1f} bp')
        print(f'  Trough: {s["trough"]["label"]} = {s["trough"]["value"]:+.1f} bp')
    display(data.to_dataframe())

## 7. Roll/Carry Heatmap

Barnes microfly carry: position i slides to i-1 after one quarter.
Positive roll = earn carry if long.

In [ ]:
# Build heatmap of rolls across all fly structures
fly_types = [StructureType.FLY_3M, StructureType.FLY_6M, StructureType.FLY_9M, StructureType.FLY_12M]
roll_data = {}
for st in fly_types:
    if st in snap.structures and snap.structures[st].labels:
        d = snap.structures[st]
        roll_data[STRUCTURE_LABELS[st]] = pd.Series(d.rolls, index=d.labels)

if roll_data:
    roll_df = pd.DataFrame(roll_data)
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(roll_df.T.values, cmap='RdYlGn', aspect='auto',
                   vmin=-max(abs(roll_df.min().min()), abs(roll_df.max().max())),
                   vmax=max(abs(roll_df.min().min()), abs(roll_df.max().max())))
    ax.set_xticks(range(len(roll_df.index)))
    ax.set_xticklabels(roll_df.index, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(roll_df.columns)))
    ax.set_yticklabels(roll_df.columns)
    for i in range(len(roll_df.columns)):
        for j in range(len(roll_df.index)):
            val = roll_df.iloc[j, i]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax, label='Roll (bp)')
    ax.set_title('Fly Roll/Carry Heatmap (bp per quarter)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No fly data available for roll heatmap')

## 8. Z-Score Heatmap

Current level vs rolling mean/std. Extreme z-scores suggest mispricings.

In [ ]:
all_types = [StructureType.SPD_3M, StructureType.SPD_6M, StructureType.SPD_12M,
             StructureType.FLY_3M, StructureType.FLY_6M, StructureType.FLY_12M,
             StructureType.DFLY_3M, StructureType.DFLY_6M]

zs_data = {}
for st in all_types:
    if st in snap.structures and snap.structures[st].labels:
        d = snap.structures[st]
        zs_data[STRUCTURE_LABELS[st]] = pd.Series(d.zscores, index=d.labels)

if zs_data:
    zs_df = pd.DataFrame(zs_data)
    fig, ax = plt.subplots(figsize=(16, 6))
    im = ax.imshow(zs_df.T.values, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
    ax.set_xticks(range(len(zs_df.index)))
    ax.set_xticklabels(zs_df.index, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(zs_df.columns)))
    ax.set_yticklabels(zs_df.columns)
    for i in range(len(zs_df.columns)):
        for j in range(len(zs_df.index)):
            val = zs_df.iloc[j, i]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7,
                       color='white' if abs(val) > 1.5 else 'black')
    plt.colorbar(im, ax=ax, label='Z-Score')
    ax.set_title('Z-Score Heatmap (60d rolling)', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Full Screener: Top Trades by Risk-Adjusted Roll

Combines roll/carry and vol to identify the best risk/reward structures.

In [ ]:
rows = []
for st, data in snap.structures.items():
    if st == StructureType.STRIP or not data.labels:
        continue
    for i, label in enumerate(data.labels):
        rows.append({
            'Structure': STRUCTURE_LABELS[st],
            'Label': label,
            'Level (bp)': data.levels[i],
            'Chg (bp)': data.changes[i],
            'Z-Score': data.zscores[i],
            'Vol (ann)': data.vols[i],
            'Roll (bp)': data.rolls[i],
            'Risk-Adj Roll': data.risk_adj_rolls[i],
        })

screener_df = pd.DataFrame(rows)
screener_df = screener_df.dropna(subset=['Risk-Adj Roll'])

print('=== TOP 10 BY POSITIVE RISK-ADJ ROLL (Long Candidates) ===')
display(screener_df.nlargest(10, 'Risk-Adj Roll').round(2))

print('\n=== TOP 10 BY NEGATIVE RISK-ADJ ROLL (Short Candidates) ===')
display(screener_df.nsmallest(10, 'Risk-Adj Roll').round(2))

print('\n=== MOST EXTREME Z-SCORES (Potential Mispricings) ===')
display(screener_df.reindex(screener_df['Z-Score'].abs().nlargest(10).index).round(2))

## 10. Kink Detection

Double flies flag kinks. Large absolute dfly values = curve irregularity.

In [ ]:
for st in [StructureType.DFLY_3M, StructureType.DFLY_6M]:
    if st not in snap.structures:
        continue
    data = snap.structures[st]
    if not data.labels:
        continue
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [2, 1]})
    x = range(len(data.labels))
    ax1.plot(x, data.levels, 'o-', color='tab:cyan', linewidth=2, markersize=6)
    for i, (lbl, lvl) in enumerate(zip(data.labels, data.levels)):
        if not np.isnan(lvl):
            ax1.annotate(f'{lvl:.1f}', (i, lvl), textcoords='offset points',
                        xytext=(0, 10), ha='center', fontsize=7, color='red')
    ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(data.labels, rotation=45, ha='right', fontsize=7)
    ax1.set_title(f'{STRUCTURE_LABELS[st]} Curve (bp)', fontweight='bold')
    ax1.grid(True, alpha=0.3)

    colors = ['tab:green' if c >= 0 else 'tab:red' for c in data.changes]
    ax2.bar(x, data.changes, color=colors, alpha=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(data.labels, rotation=45, ha='right', fontsize=7)
    ax2.set_title('1d Change (bp)', fontweight='bold')
    ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()